In [ ]:
options(encoding = "UTF-8")
Sys.setlocale("LC_ALL", "pl_PL.UTF-8")

# packages loader
use_package <- function(pkg) {
  if (!require(pkg, character.only = TRUE)) {
    install.packages(pkg)
    library(pkg, character.only = TRUE)
  }
}

use_package("MASS")


# Exercise 1
## About methods
- about methods - how they work
## About data
Geyser data set
- describe eruption duration ($duration (min)) and and time between eruptions ($waiting (?)) from (1985-1990)
- 
## My interpretation 
### Densities
#### Methodology
#### Resutls 
### Ratio
#### Methodology
- I skip *BCV* method because it was to *flat*. 
#### Results
-- 
porównac gładkość i rozdzielczość - z czego wynika, że pewne estymacje zwiększają przyrost a inne nie  

In [ ]:
help(geyser)

In [ ]:
# methods
data_1 = geyser$duration
ucv(data_1)
bcv(data_1)
width.SJ(data_1)
width.SJ(data_1, method="dpi")

## Densities
# kernel densities - windows width
bw_ucv <- ucv(data_1)
bw_bcv <- bcv(data_1)
bw_sj  <- width.SJ(data_1)
bw_dpi <- width.SJ(data_1, method = "dpi")
# kernel densities - pdfs
d_ucv <- density(data_1, bw = bw_ucv)
d_bcv <- density(data_1, bw = bw_bcv)
d_sj  <- density(data_1, bw = bw_sj)
d_dpi <- density(data_1, bw = bw_dpi)


## plots
# histogram
hist(data_1, breaks = 30, probability = TRUE, col = "lightgray", main = "Histogram + KDE", xlab = "Duration (min)")
# densities - "density" function uses by default gaussian kernel 
lines(d_ucv, col = "blue", lwd = 2)
lines(d_bcv, col = "green", lwd = 2)
lines(d_sj,  col = "red", lwd = 2)
lines(d_dpi, col = "purple", lwd = 2)
# legend
legend("topright", legend = c("UCV", "BCV", "SJ", "DPI"), col = c("blue", "green", "red", "purple"), lwd = 2)

## Maximum comparison
# local maximum function
local_max <- function(pdf, domain) {
  range <- which(pdf$x >= domain[1] & pdf$x <= domain[2])
  max(pdf$y[range])
}
# Dataframe
results <- data.frame(
  Method = c("UCV", "SJ", "DPI"),
  Left   = c(local_max(d_ucv, c(1,3)),
             local_max(d_sj,  c(1,3)),
             local_max(d_dpi, c(1,3))),
  Right  = c(local_max(d_ucv, c(3,5)),
             local_max(d_sj,  c(3,5)),
             local_max(d_dpi, c(3,5)))
)
results$Ratio = results$Left / results$Right
print(results)

# Exercise 2

In [ ]:
# Helper functions 

# random probe claw density 
rclaw = function(n) {
    # first component
    comp = 0.5 * rnorm(n, mean = 0, sd=1)
    # rest components
    for (i in 0:4) {
        comp = comp + 0.1*rnorm(n, mean=i/2-1, sd=0.1^2)
    }
    return(comp)
}

# function for analysis densities 
analyze_densities = function(data){
    ## Densities
    # kernel densities - windows width
    bw_ucv <- ucv(data)
    bw_bcv <- bcv(data)
    bw_sj  <- width.SJ(data)
    bw_dpi <- width.SJ(data, method = "dpi")
    # kernel densities - pdfs
    d_ucv <- density(data, bw = bw_ucv)
    d_bcv <- density(data, bw = bw_bcv)
    d_sj  <- density(data, bw = bw_sj)
    d_dpi <- density(data, bw = bw_dpi)

    ## plots
    # histogram
    hist(data, breaks = 30, probability = TRUE, col = "lightgray", main = "Histogram + KDE", xlab = "X value")
    # densities - "density" function uses by default gaussian kernel 
    lines(d_ucv, col = "blue", lwd = 2)
    lines(d_bcv, col = "green", lwd = 2)
    lines(d_sj,  col = "red", lwd = 2)
    lines(d_dpi, col = "purple", lwd = 2)
    # legend
    legend("topright", legend = c("UCV", "BCV", "SJ", "DPI"), col = c("blue", "green", "red", "purple"), lwd = 2)
    
    ## Windows results 
    return(data.frame(
        bw_ucv = bw_ucv,
        bw_bcv = bw_bcv,
        bw_sj  = bw_sj,
        bw_dpi = bw_dpi
    ))
}

In [ ]:
# Results
n_vector = c(100, 200, 300, 1000)
df = data.frame()
# analyze
for (n in n_vector){
    probe = rclaw(n)
    loc_df = analyze_densities(probe) 
    # adding index column (n)
    loc_df$n = n
    # adding to global df
    df = rbind(df, loc_df)
}
# setting columns right
df = df[, c("n", "bw_ucv", "bw_bcv", "bw_sj", "bw_dpi")]

print(df)